# Module 11 Lab - Hyperparameter Tuning & AutoML**Objective:** To learn how to optimize model performance by tuning **hyperparameters** and to get an introduction to the powerful concept of **Automated Machine Learning (AutoML)**.**In this lab, you will write the code to perform Grid Search and Random Search to find the best hyperparameters for a model.**

## Part 1: What are Hyperparameters?**Concept:** In machine learning, there are two types of parameters:1.  **Model Parameters:** These are parameters that the model learns from the data during training. For example, the coefficients in a Linear Regression model.2.  **Hyperparameters:** These are parameters that are **set before training begins**. They are not learned from the data; instead, they are choices we make about the model's structure or how it learns.     *   *Examples:* The `n_estimators` in a Random Forest (how many trees to build), the `max_depth` of a Decision Tree (how deep it can grow), or the `C` regularization parameter in a Logistic Regression.Finding the right hyperparameters can have a huge impact on a model's performance. **Hyperparameter tuning** is the process of systematically searching for the best combination of these settings.

## Part 2: SetupWe will use the Iris dataset and a `RandomForestClassifier`, which has several important hyperparameters we can tune.

In [6]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# A baseline model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"Accuracy of baseline Random Forest: {accuracy_baseline:.2%}")

Accuracy of baseline Random Forest: 100.00%


## Part 3: Grid Search**Concept:** Grid Search is the most straightforward tuning method. You define a "grid" of hyperparameter values you want to try, and the algorithm exhaustively trains and evaluates a model for **every possible combination**.*   **Pro:** It's guaranteed to find the best combination within the grid.*   **Con:** It can be very slow and computationally expensive if the grid is large.

### Task 1: Perform a Grid Search**Your Task:** Use `GridSearchCV` from `sklearn.model_selection` to search for the best `n_estimators` and `max_depth` for our Random Forest.

In [7]:
from sklearn.model_selection import GridSearchCV
import numpy as np

# Define the grid of hyperparameters to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None]
}

# Create a GridSearchCV instance
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit the grid search to the data
grid_search.fit(X_train, y_train)

# Print the best parameters and the best score
print(f"Best Parameters found by Grid Search: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameters found by Grid Search: {'max_depth': 5, 'n_estimators': 100}
Best cross-validated score: 94.29%


## Part 4: Random Search**Concept:** Random Search is often more efficient than Grid Search. Instead of trying every combination, it randomly samples a fixed number of combinations from the hyperparameter space. *   **Pro:** It's much faster and can explore a wider range of values.*   **Con:** It's not guaranteed to find the absolute best combination, but it often finds a very good one much more quickly.

### Task 2: Perform a Random Search**Your Task:** Use `RandomizedSearchCV` to perform a random search over a larger hyperparameter space.

In [3]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Define the distribution of hyperparameters to sample from
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start = 50, stop = 500, num = 10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# Create a RandomizedSearchCV instance
random_search = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42), param_distributions=param_dist, n_iter=10, cv=5, n_jobs=-1, verbose=2, random_state=42)

# Fit the random search to the data
random_search.fit(X_train, y_train)

# Print the best parameters and the best score
print(f"Best Parameters found by Random Search: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters found by Random Search: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 94.29%


## Part 5: Introduction to AutoML with AutoGluon**Concept:** AutoML takes hyperparameter tuning to the next level. It automates the entire ML workflow, including:*   Data preprocessing*   Feature engineering*   Model selection (trying many different types of models)*   Hyperparameter tuning*   Ensemble creation**AutoGluon** is a popular and easy-to-use AutoML library. With just a few lines of code, it can train and tune dozens of models and create a powerful ensemble.**This part is fully coded.** Your task is to run it and see the power of AutoML. Note that it may take a few minutes to run.

In [11]:
import sys
import os
import logging

# Redirect stdout and stderr to suppress all autogluon output
old_stdout = sys.stdout
old_stderr = sys.stderr
sys.stdout = open(os.devnull, 'w')
sys.stderr = open(os.devnull, 'w')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

import numpy as np
from autogluon.tabular import TabularPredictor

# AutoGluon requires the data in a single DataFrame with the target column.
train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train

test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test

# Create and train with quiet settings
predictor = TabularPredictor(label='species', eval_metric='accuracy').fit(
    train_data=train_data_ag,
    time_limit=60,
    verbosity=0,
    presets='medium',
    excluded_model_types=['FASTAI', 'NN_TORCH', 'CAT', 'XGB', 'GBM']
)

# Restore stdout/stderr
sys.stdout.close()
sys.stderr.close()
sys.stdout = old_stdout
sys.stderr = old_stderr

# Now print only what we want
print(predictor.leaderboard(test_data_ag, silent=True))

                 model  score_test  score_val eval_metric  pred_time_test  \
0       ExtraTreesGini         1.0   0.904762    accuracy        0.094294   
1     RandomForestGini         1.0   0.857143    accuracy        0.095518   
2       ExtraTreesEntr         1.0   0.904762    accuracy        0.100095   
3     RandomForestEntr         1.0   0.857143    accuracy        0.104957   
4  WeightedEnsemble_L2         1.0   0.904762    accuracy        0.124099   

   pred_time_val  fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0       0.106381  1.298569                 0.094294                0.106381   
1       0.091354  1.389960                 0.095518                0.091354   
2       0.047761  1.206581                 0.100095                0.047761   
3       0.092606  1.303610                 0.104957                0.092606   
4       0.107497  1.343130                 0.029805                0.001116   

   fit_time_marginal  stack_level  can_infer  fit_order  
0   

## 📝 Knowledge Check**Instructions:** Answer the following questions in this markdown cell.1.  **What is the main difference be
tween a model parameter and a hyperparameter?**2.  **When would you choose to use Grid Search over Random Search, and vice-versa?**3.  **Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?****[ENTER YOUR ANSWERS HERE]**

1. **What is the main difference between a model parameter and a hyperparameter?**

Model parameters are values that the model learns from the training data during the training process - things like the weights in a neural network or the coefficients in a linear regression. Hyperparameters, on the other hand, are settings we choose before training starts. They control how the model learns but aren't directly learned from the data. Examples include how many trees to grow in a Random Forest (n_estimators) or how deep each tree can go (max_depth).

2. **When would you choose to use Grid Search over Random Search, and vice-versa?**

I would use Grid Search when I have a small, well-defined search space and want to guarantee finding the best combination within that space. It's good when computation isn't a constraint. However, it gets really slow as the grid grows. Random Search is better when I want to explore a larger hyperparameter space more efficiently - especially when I'm not sure which hyperparameters matter most. It randomly samples combinations, so it can find a good solution much faster even with a bigger search space. The tradeoff is that it's not guaranteed to find the absolute best, but in practice it often finds a very good one.

3. **Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?**

The best model on the AutoGluon leaderboard was the **WeightedEnsemble_L2**, which combined the predictions from the best individual models (ExtraTreesGini and ExtraTreesEntr with 90.48% validation accuracy). Both ExtraTreesGini and ExtraTreesEntr achieved the same score of 0.904762 on validation.

AutoML is powerful because it automates a lot of the manual work involved in building a good ML model. Instead of me having to manually pick a model type, tune hyperparameters, and engineer features, AutoGluon automatically tries multiple model types (Random Forest, Extra Trees, XGBoost, etc.), tunes their hyperparameters, and even creates an ensemble of the best models. It also handles data preprocessing and feature engineering automatically. This saves a ton of time and often produces better results than what I could tune manually, since it explores way more combinations than I'd realistically have time to test on my own.